#### 전략 1. A/B 클래스 전용 비율 피처 생성

- `연체입금원금_B0M` / `전체입금금액`, `쇼핑_도소매_이용금액` / `총이용금액` 등 비율로 전환
- 특히 소액을 쓰거나 일정한 소비패턴을 보이는 A/B 클래스의 특성을 수치로 잡아내기 쉬움

In [2]:
import pandas as pd
import numpy as np
import os

In [20]:
# 데이터셋 경로 리스트
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

In [22]:
# 20개 컬럼명을 담은 리스트
columns = pd.read_csv('./result/컬럼_감소_과정.csv')
columns_20 = columns['20'].dropna().tolist()

In [23]:
columns_20

['이용금액_R3M_신용체크',
 '_2순위카드이용금액',
 '최대이용금액_일시불_R12M',
 '연체입금원금_B0M',
 '_2순위업종_이용금액',
 '_3순위업종_이용금액',
 '_1순위교통업종_이용금액',
 '이용건수_신용_R12M',
 '이용금액_오프라인_B0M',
 '이용건수_오프라인_R6M',
 '_1순위업종_이용금액',
 '정상입금원금_B2M',
 '정상입금원금_B5M',
 '이용금액_오프라인_R6M',
 '쇼핑_도소매_이용금액',
 '_3순위쇼핑업종_이용금액',
 '청구금액_R6M',
 '청구금액_B0',
 '잔액_일시불_B0M',
 '평잔_일시불_3M']

In [24]:
# 병합할 데이터프레임 리스트
df_list = []

for path in paths:
    if os.path.exists(path):
        df = pd.read_parquet(path)
        common_cols = list(set(df.columns) & set(columns_20))
        if common_cols:
            df_list.append(df[common_cols])

# 좌우 병합 및 중복 제거
merged_df = pd.concat(df_list, axis=1)
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

# 저장
merged_df.to_csv('./result/merged_df_20cols.csv', index=False, encoding='utf-8-sig')

In [ ]:
# 상위20개컬럼.csv로 이름 변경
df = pd.read_csv("./result/상위20개컬럼.csv")
df

,이용금액_R3M_신용체크,_2순위카드이용금액,_1순위업종_이용금액,정상입금원금_B5M,이용금액_오프라인_B0M,_2순위업종_이용금액,최대이용금액_일시불_R12M,연체입금원금_B0M,_3순위쇼핑업종_이용금액,이용건수_오프라인_R6M,정상입금원금_B2M,_3순위업종_이용금액,_1순위교통업종_이용금액,이용건수_신용_R12M,쇼핑_도소매_이용금액,이용금액_오프라인_R6M,청구금액_B0,청구금액_R6M,평잔_일시불_3M,잔액_일시불_B0M
0,196,0,1928,9205,4043,1408,4906,8104,0,54,16125,672,200,165,0,11097,12226,88693,1791,998
1,13475,0,2158,2546,3980,2083,10407,826,315,95,2420,1659,1215,204,645,18638,5834,16861,3761,2565
2,23988,0,16924,16949,4524,1539,10112,9364,924,76,14448,1362,1362,148,1038,29192,21866,165221,6796,5312
3,3904,0,2405,8418,3975,2284,3075,10923,0,39,13043,774,208,105,0,18056,16356,127371,772,730
4,1190,0,0,0,0,0,62,0,0,0,0,0,0,-1,0,787,0,155,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,10755,0,0,0,0,0,0,19,0,0,0,0,0,0,0,0,0,0,0,0
2399996,27636,0,5608,21831,4676,1810,47684,3756,346,57,10764,1547,632,219,1496,60373,14402,99849,9424,3351
2399997,23187,0,3634,3269,4516,2315,10212,1703,305,69,6106,2302,1925,170,0,32036,5731,41073,2998,2524
2399998,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [3]:
# 1. 상위 20개 컬럼 데이터 불러오기
# df = pd.read_csv("./result/상위20개컬럼.csv")

# 2. Segment 컬럼이 들어있는 파일에서 Segment만 추출
df_segment = pd.read_parquet('./data/train/1_회원정보_train.parquet', columns=['Segment'])

# 3. 인덱스 기준으로 병합 (row 수가 같다는 전제하에)
# df["Segment"] = df_segment["Segment"]

In [14]:
# 결과 확인
print(df_segment.value_counts())
print(df_segment.value_counts(normalize=True))

Segment
E          1922052
D           349242
C           127590
A              972
B              144
Name: count, dtype: int64
Segment
E          0.800855
D          0.145517
C          0.053163
A          0.000405
B          0.000060
Name: proportion, dtype: float64


In [35]:
df.to_csv('./result/상위20개컬럼.csv', index=False, encoding='utf-8-sig')

### 피처 생성

In [ ]:
# 0으로 나누는 상황 방지를 위한 함수
def safe_divide(a, b):
    return np.where(b == 0, np.nan, a / b)

# 비율 피처 생성
df["연체비율_B0M"] = safe_divide(df["연체입금원금_B0M"], df["청구금액_B0"])
df["일시불잔액비율"] = safe_divide(df["잔액_일시불_B0M"], df["청구금액_B0"])
df["정상입금안정성"] = safe_divide(df["정상입금원금_B5M"], df["정상입금원금_B2M"])
df["쇼핑소비비율"] = safe_divide(df["쇼핑_도소매_이용금액"], df["이용금액_오프라인_R6M"])
df["카드소비편중도"] = safe_divide(df["_1순위업종_이용금액"], df["_2순위업종_이용금액"])
df["총카드소비편중도"] = safe_divide(
    df["_1순위업종_이용금액"],
    df["_2순위업종_이용금액"] + df["_3순위업종_이용금액"]
)

# 결과 확인 (선택)
print(df[[
    "연체비율_B0M", "일시불잔액비율", "정상입금안정성",
    "쇼핑소비비율", "카드소비편중도", "총카드소비편중도"
]].describe())

           연체비율_B0M       일시불잔액비율       정상입금안정성        쇼핑소비비율       카드소비편중도  \
count  1.852372e+06  1.852372e+06  1.716812e+06  1.943685e+06  1.487265e+06   
mean   6.213391e-01  7.701935e-01  1.091746e+00  3.951842e-02  3.672904e+00   
std    4.264953e+00  7.303440e-01  9.856657e-01  6.947312e-01  1.205348e+01   
min    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00   
25%    0.000000e+00  3.832403e-01  6.768012e-01  0.000000e+00  1.347934e+00   
50%    3.139099e-02  6.585219e-01  9.740188e-01  1.437517e-02  1.972635e+00   
75%    3.679774e-01  9.785408e-01  1.346980e+00  3.048462e-02  3.424920e+00   
max    3.902222e+02  3.478469e+01  9.432143e+01  3.970000e+02  4.395000e+03   

           총카드소비편중도  
count  1.487265e+06  
mean   2.721559e+00  
std    1.197380e+01  
min    5.002529e-01  
25%    8.704166e-01  
50%    1.305520e+00  
75%    2.265707e+00  
max    4.395000e+03  


In [31]:
# 저장 (필요한 경우)
df.to_csv("./result/비율_피처생성결과.csv", index=False, encoding='utf-8-sig')

In [32]:
df

,이용금액_R3M_신용체크,_2순위카드이용금액,_1순위업종_이용금액,정상입금원금_B5M,이용금액_오프라인_B0M,_2순위업종_이용금액,최대이용금액_일시불_R12M,연체입금원금_B0M,_3순위쇼핑업종_이용금액,이용건수_오프라인_R6M,...,청구금액_B0,청구금액_R6M,평잔_일시불_3M,잔액_일시불_B0M,연체비율_B0M,일시불잔액비율,정상입금안정성,쇼핑소비비율,카드소비편중도,총카드소비편중도
0,196,0,1928,9205,4043,1408,4906,8104,0,54,...,12226,88693,1791,998,0.662850,0.081629,0.570853,0.000000,1.369318,0.926923
1,13475,0,2158,2546,3980,2083,10407,826,315,95,...,5834,16861,3761,2565,0.141584,0.439664,1.052066,0.034607,1.036006,0.576697
2,23988,0,16924,16949,4524,1539,10112,9364,924,76,...,21866,165221,6796,5312,0.428245,0.242934,1.173104,0.035558,10.996751,5.833850
3,3904,0,2405,8418,3975,2284,3075,10923,0,39,...,16356,127371,772,730,0.667828,0.044632,0.645404,0.000000,1.052977,0.786462
4,1190,0,0,0,0,0,62,0,0,0,...,0,155,0,0,NaN,NaN,NaN,0.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,10755,0,0,0,0,0,0,19,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2399996,27636,0,5608,21831,4676,1810,47684,3756,346,57,...,14402,99849,9424,3351,0.260797,0.232676,2.028149,0.024779,3.098343,1.670539
2399997,23187,0,3634,3269,4516,2315,10212,1703,305,69,...,5731,41073,2998,2524,0.297156,0.440412,0.535375,0.000000,1.569762,0.787091
2399998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
df[[
    "연체비율_B0M", "일시불잔액비율", "정상입금안정성",
    "쇼핑소비비율", "카드소비편중도", "총카드소비편중도"
]]

,연체비율_B0M,일시불잔액비율,정상입금안정성,쇼핑소비비율,카드소비편중도,총카드소비편중도
0,0.662850,0.081629,0.570853,0.000000,1.369318,0.926923
1,0.141584,0.439664,1.052066,0.034607,1.036006,0.576697
2,0.428245,0.242934,1.173104,0.035558,10.996751,5.833850
3,0.667828,0.044632,0.645404,0.000000,1.052977,0.786462
4,NaN,NaN,NaN,0.000000,NaN,NaN
...,...,...,...,...,...,...
2399995,NaN,NaN,NaN,NaN,NaN,NaN
2399996,0.260797,0.232676,2.028149,0.024779,3.098343,1.670539
2399997,0.297156,0.440412,0.535375,0.000000,1.569762,0.787091
2399998,NaN,NaN,NaN,NaN,NaN,NaN
